In [1]:
import pandas as pd

# Load the evaluation results
csv_path = "asr_evaluation_results.csv"
df = pd.read_csv(csv_path)

# Expand pandas display limits so we can read long transcriptions easily
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 50)

# Helper function to print samples cleanly for your thesis notes
def print_thesis_examples(dataframe, title, num_samples=5):
    print("=" * 80)
    print(f" CATEGORY: {title}")
    print("=" * 80)
    
    # Handle cases where the slice has fewer examples than requested
    samples = dataframe.head(num_samples)
    if samples.empty:
        print("No samples found matching these criteria.")
        return
        
    for idx, row in samples.iterrows():
        print(f"ID: {row['id']}")
        print(f"  [REF] Ground Truth : {row['ground_truth']}")
        print(f"  [RAW] Greedy Pred  : {row['greedy_pred']}")
        print(f"  [KNL] KenLM Pred   : {row['kenlm_pred']}")
        print(f"  Metrics -> Greedy WER: {row['greedy_wer']:.2f} | KenLM WER: {row['kenlm_wer']:.2f} | Delta: {row['wer_improvement']:.2f}")
        print(f"  Alignments -> KenLM Subs: {row['kenlm_substitutions']} | Ins: {row['kenlm_insertions']} | Del: {row['kenlm_deletions']}")
        print("-" * 80)

## Best wins

In [2]:
# Filter for cases where KenLM significantly improved the text (positive delta)
# and achieved a perfect or near-perfect score
best_wins = df[df['wer_improvement'] > 0.1].sort_values(by='wer_improvement', ascending=False)

# Focus heavily on "perfect saves" (Greedy had errors, KenLM achieved 0% WER)
perfect_saves = df[(df['greedy_wer'] > 0) & (df['kenlm_wer'] == 0)].sort_values(by='greedy_wer', ascending=False)

print_thesis_examples(perfect_saves, "PERFECT SAVES (KenLM completely fixed acoustic errors)", num_samples=5)

 CATEGORY: PERFECT SAVES (KenLM completely fixed acoustic errors)
ID: sample_1687
  [REF] Ground Truth : morna noqogħdu għand omm ir-raġel flimkien ma' zijuh
  [RAW] Greedy Pred  : morna noqogħdu għandhomm ir-raġel flimkien maz zijuh
  [KNL] KenLM Pred   : morna noqogħdu għand omm ir-raġel flimkien ma' zijuh
  Metrics -> Greedy WER: 0.38 | KenLM WER: 0.00 | Delta: 0.38
  Alignments -> KenLM Subs: 0 | Ins: 0 | Del: 0
--------------------------------------------------------------------------------
ID: sample_248
  [REF] Ground Truth : grazzi mill-ġdid lil ruth castillo bl-aġġornament tal-aħbarijiet tat-tmienja
  [RAW] Greedy Pred  : grazzi mill-ġdid il-ruth castello bl-aġġornament tal-aħbarijiet tat-tmienja
  [KNL] KenLM Pred   : grazzi mill-ġdid lil ruth castillo bl-aġġornament tal-aħbarijiet tat-tmienja
  Metrics -> Greedy WER: 0.38 | KenLM WER: 0.00 | Delta: 0.38
  Alignments -> KenLM Subs: 0 | Ins: 0 | Del: 0
---------------------------------------------------------------------------

In [3]:
# Filter for cases where KenLM made the transcription worse (negative delta)
worst_failures = df[df['wer_improvement'] < -0.1].sort_values(by='wer_improvement', ascending=True)

print_thesis_examples(worst_failures, "CRITICAL DEGRADATIONS (KenLM broke or over-corrected sentences)", num_samples=5)

 CATEGORY: CRITICAL DEGRADATIONS (KenLM broke or over-corrected sentences)
ID: sample_455
  [REF] Ground Truth : filwaqt illi fil-fil-fl-eku fid-diwi fl-isfond hemm
  [RAW] Greedy Pred  : filwaqt illi fil-fil-fl-eku fid-diwi fl-isfond hemm
  [KNL] KenLM Pred   : filwaqt illi fil-file u fid-diwi fl-isfond hemm
  Metrics -> Greedy WER: 0.00 | KenLM WER: 0.33 | Delta: -0.33
  Alignments -> KenLM Subs: 1 | Ins: 1 | Del: 0
--------------------------------------------------------------------------------
ID: sample_498
  [REF] Ground Truth : kull reazzjoni kull diskussjoni li ssir fuqha f' bar
  [RAW] Greedy Pred  : kull reazzjoni kull diskussjoni li ssir fuqha f' bar
  [KNL] KenLM Pred   : kull reazzjoni u kull diskussjoni li ssir fuqha f'bar
  Metrics -> Greedy WER: 0.00 | KenLM WER: 0.33 | Delta: -0.33
  Alignments -> KenLM Subs: 1 | Ins: 1 | Del: 1
--------------------------------------------------------------------------------
ID: sample_24
  [REF] Ground Truth : x' inhi ħrafa hekk kif t

In [4]:
# Look for aggressive deletions (KenLM deleted words that the greedy baseline captured)
high_deletions = df[df['kenlm_deletions'] > df['greedy_deletions']].sort_values(by='kenlm_deletions', ascending=False)

# Look for aggressive insertions (KenLM hallucinated filler words or phrases)
high_insertions = df[df['kenlm_insertions'] > df['greedy_insertions']].sort_values(by='kenlm_insertions', ascending=False)

print_thesis_examples(high_deletions, "DIAGNOSTIC: High KenLM Deletions (Is Beta penalizing word counts too heavily?)", num_samples=3)
print_thesis_examples(high_insertions, "DIAGNOSTIC: High KenLM Insertions (Is Beta rewarding word generation too heavily?)", num_samples=3)

 CATEGORY: DIAGNOSTIC: High KenLM Deletions (Is Beta penalizing word counts too heavily?)
ID: sample_190
  [REF] Ground Truth : mill-viljakk billi billi jipprova jikkonvinċi u jipperswadi lil karattri oħrajn b' imġiba falza
  [RAW] Greedy Pred  : mill-viljakk billi billi jipprova jikkonvinċi u jipperswadi lil karattri oħrajn bl-imġiba falza
  [KNL] KenLM Pred   : mill-viljakkbilli jipprova jikkonvinċi u jipperswadi lil karattri oħrajn bl-imġiba falza
  Metrics -> Greedy WER: 0.15 | KenLM WER: 0.38 | Delta: -0.23
  Alignments -> KenLM Subs: 2 | Ins: 0 | Del: 3
--------------------------------------------------------------------------------
ID: sample_230
  [REF] Ground Truth : l-awtur għandu l-opinjoni tiegħu xi u ħa jgħid x'jaħseb hu jew ħa jgħid
  [RAW] Greedy Pred  : l-awtur għandu l-opinjoni tiegħu een ħa jgħit x' jaħseb hu  ħa għit
  [KNL] KenLM Pred   : l-awtur għandu l-opinjoni tiegħu eiena jgħid x'jaħseb hu ħa għit
  Metrics -> Greedy WER: 0.54 | KenLM WER: 0.38 | Delta: 0.15
  

In [5]:
total_samples = len(df)
improved = len(df[df['wer_improvement'] > 0])
worsened = len(df[df['wer_improvement'] < 0])
unchanged = len(df[df['wer_improvement'] == 0])

summary_data = {
    "Impact Category": ["Improved by KenLM", "Worsened by KenLM", "No Change / Identical"],
    "Sample Count": [improved, worsened, unchanged],
    "Percentage (%)": [(improved/total_samples)*100, (worsened/total_samples)*100, (unchanged/total_samples)*100]
}

summary_df = pd.DataFrame(summary_data)
print("### QUANTITATIVE OVERVIEW TABLE ###")
display(summary_df.round(2))

# Average error profile comparison
print("\n### AVERAGE WORD-LEVEL ALIGNMENT PROFILE ###")
alignment_comparison = df[['greedy_substitutions', 'kenlm_substitutions', 
                           'greedy_insertions', 'kenlm_insertions', 
                           'greedy_deletions', 'kenlm_deletions']].mean().to_frame(name='Dataset Mean')
display(alignment_comparison.round(3))

### QUANTITATIVE OVERVIEW TABLE ###


,Impact Category,Sample Count,Percentage (%)
0,Improved by KenLM,302,16.04
1,Worsened by KenLM,123,6.53
2,No Change / Identical,1458,77.43



### AVERAGE WORD-LEVEL ALIGNMENT PROFILE ###


,Dataset Mean
greedy_substitutions,0.853
kenlm_substitutions,0.695
greedy_insertions,0.101
kenlm_insertions,0.122
greedy_deletions,0.148
kenlm_deletions,0.131


In [6]:
# Filter for instances where the raw baseline model got 100% accuracy (WER = 0)
raw_perfect = df[df['greedy_wer'] == 0]

percent_perfect = (len(raw_perfect) / len(df)) * 100

print("=" * 80)
print(f" ANALYSIS: RAW MODEL BEST PERFORMANCE")
print("=" * 80)
print(f"Total samples with 0% Greedy WER: {len(raw_perfect)} out of {len(df)} ({percent_perfect:.2f}%)")
print("-" * 80)

# Display a subset of these perfect raw transcriptions
# (Note: KenLM will likely also be 0 here, or occasionally worse if it over-corrected)
print_thesis_examples(raw_perfect.sort_values(by='ground_truth', key=lambda x: x.str.len(), ascending=False), 
                      "RAW MODEL PERFECT TRANSCRIPTIONS (Longest Flawless Sentences)", 
                      num_samples=5)

 ANALYSIS: RAW MODEL BEST PERFORMANCE
Total samples with 0% Greedy WER: 1056 out of 1883 (56.08%)
--------------------------------------------------------------------------------
 CATEGORY: RAW MODEL PERFECT TRANSCRIPTIONS (Longest Flawless Sentences)
ID: sample_583
  [REF] Ground Truth : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  [RAW] Greedy Pred  : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  [KNL] KenLM Pred   : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  Metrics -> Greedy WER: 0.00 | KenLM WER: 0.00 | Delta: 0.00
  Alignments -> KenLM Subs: 0 | Ins: 0 | Del: 0
--------------------------------------------------------------------------------
ID: sample_631
  [REF] Ground Truth : ikollhom e il-il-biex tidħo

In [7]:
# Filter for cases where the raw model completely failed (WER >= 0.80)
# Sort by greedy_wer descending to see the absolute worst first
raw_collapses = df[df['greedy_wer'] >= 0.80].sort_values(by='greedy_wer', ascending=False)

percent_collapses = (len(raw_collapses) / len(df)) * 100

print("=" * 80)
print(f" ANALYSIS: RAW MODEL WORST PERFORMANCE (TOTAL COLLAPSES)")
print("=" * 80)
print(f"Total samples with >= 80% Greedy WER: {len(raw_collapses)} out of {len(df)} ({percent_collapses:.2f}%)")
print("-" * 80)

# Print the top 5 absolute worst failures of the raw model
print_thesis_examples(raw_collapses, 
                      "RAW MODEL TOTAL COLLAPSES (Worst Baseline Transcriptions)", 
                      num_samples=5)

 ANALYSIS: RAW MODEL WORST PERFORMANCE (TOTAL COLLAPSES)
Total samples with >= 80% Greedy WER: 7 out of 1883 (0.37%)
--------------------------------------------------------------------------------
 CATEGORY: RAW MODEL TOTAL COLLAPSES (Worst Baseline Transcriptions)
ID: sample_344
  [REF] Ground Truth : kienet serje ta' istruzzjonijiet ta' litteralment liem kordi trid
  [RAW] Greedy Pred  : iġiene seja ta istruzzjoijit talu letelmn ln od t tid
  [KNL] KenLM Pred   : jien ej ta istruzzjonijiet alueelta o d i i
  Metrics -> Greedy WER: 1.11 | KenLM WER: 0.89 | Delta: 0.22
  Alignments -> KenLM Subs: 8 | Ins: 0 | Del: 0
--------------------------------------------------------------------------------
ID: sample_648
  [REF] Ground Truth : għalfejn qed jgħixu
  [RAW] Greedy Pred  : għal fejn qed jimxu
  [KNL] KenLM Pred   : għal fejn qed jimxu
  Metrics -> Greedy WER: 1.00 | KenLM WER: 1.00 | Delta: 0.00
  Alignments -> KenLM Subs: 2 | Ins: 1 | Del: 0
----------------------------------------

In [ ]:
# Filter for all samples that had any errors at all (WER > 0)
imperfect_raw = df[df['greedy_wer'] > 0]

# Calculate the proportion of errors coming from Subs, Ins, and Dels
total_raw_errors = imperfect_raw['greedy_substitutions'].sum() + imperfect_raw['greedy_insertions'].sum() + imperfect_raw['greedy_deletions'].sum()

sub_pct = (imperfect_raw['greedy_substitutions'].sum() / total_raw_errors) * 100
ins_pct = (imperfect_raw['greedy_insertions'].sum() / total_raw_errors) * 100
del_pct = (imperfect_raw['greedy_deletions'].sum() / total_raw_errors) * 100

print("### RAW MODEL ERROR BEHAVIOR PROFILE ###")
print(f"When the raw model makes mistakes, the error distribution is:")
print(f"  - Substitutions (Wrong words): {sub_pct:.2f}%")
print(f"  - Deletions (Skipped words):    {del_pct:.2f}%")
print(f"  - Insertions (Extra words):     {ins_pct:.2f}%")
print("-" * 80)
print("💡 Thesis Hint: If Deletions are high, your model struggles with fast speech/audio drops.")
print("               If Substitutions are high, it struggles with accents, homophones, or background noise.")

### RAW MODEL ERROR BEHAVIOR PROFILE ###
When the raw model makes mistakes, the error distribution is:
  - Substitutions (Wrong words): 77.40%
  - Deletions (Skipped words):    13.45%
  - Insertions (Extra words):     9.16%
--------------------------------------------------------------------------------
💡 Thesis Hint: If Deletions are high, your model struggles with fast speech/audio drops.
               If Substitutions are high, it struggles with accents, homophones, or background noise.
